# 00 — Proof of Concept: Degradazione EnCodec

Questo notebook verifica l'intera catena **audio → EnCodec → audio degradato** prima di costruire la pipeline di training.
Obiettivo: quantificare quanto EnCodec a 6 kbps degrada un file audio e visualizzare la perdita sul mel spectrogram.

---

## Cella 1 — Installazione dipendenze

Installiamo `encodec` (Meta, compressione neurale), `torchaudio` (I/O e DSP), `librosa` (analisi spettrale) e `matplotlib` (visualizzazioni).
Su Colab T4 queste librerie non sono pre-installate nella versione richiesta.

In [ ]:
!pip install -q encodec torchaudio librosa matplotlib

## Cella 2 — Caricamento e preprocessing audio

Carichiamo un file audio dall'utente tramite `files.upload()` di Colab.
Lo portiamo a **24 kHz mono** (frequenza di campionamento richiesta da EnCodec 24kHz) e
estraiamo **8 secondi dal centro** per avere un segmento rappresentativo ed evitare silenzio iniziale/finale.

In [ ]:
import io
import torch
import torchaudio
import torchaudio.transforms as T
from google.colab import files

TARGET_SR = 24_000
SEGMENT_SECONDS = 8

# --- upload ---
uploaded = files.upload()
filename = list(uploaded.keys())[0]
audio_bytes = uploaded[filename]

waveform, sr = torchaudio.load(io.BytesIO(audio_bytes))

# mono
if waveform.shape[0] > 1:
    waveform = waveform.mean(dim=0, keepdim=True)

# resample a 24 kHz
if sr != TARGET_SR:
    waveform = T.Resample(sr, TARGET_SR)(waveform)

# estrai 8 secondi dal centro
total_samples = waveform.shape[-1]
seg_samples = TARGET_SR * SEGMENT_SECONDS
start = max(0, (total_samples - seg_samples) // 2)
waveform = waveform[:, start : start + seg_samples]

print(f"File: {filename}")
print(f"Shape: {waveform.shape}  |  SR: {TARGET_SR} Hz  |  Durata: {waveform.shape[-1]/TARGET_SR:.2f}s")

## Cella 3 — Degradazione con EnCodec 24kHz a 6 kbps

Carichiamo il modello **EnCodec 24kHz** di Meta e impostiamo il bandwidth a **6 kbps** (4 codebook RVQ).
Il ciclo encode → decode introduce la distorsione da compressione neurale che vogliamo imparare a correggere con la UNet.
Salviamo sia `clean` che `degraded` come numpy array per le analisi successive.

In [ ]:
from encodec import EncodecModel
from encodec.utils import convert_audio

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# carica modello EnCodec 24kHz
model = EncodecModel.encodec_model_24khz()
model.set_target_bandwidth(6.0)   # 6 kbps → 4 codebook
model = model.to(device)
model.eval()

# prepara tensore [batch, channels, samples]
wav = convert_audio(waveform, TARGET_SR, model.sample_rate, model.channels)
wav_input = wav.unsqueeze(0).to(device)  # [1, 1, T]

with torch.no_grad():
    encoded_frames = model.encode(wav_input)
    decoded = model.decode(encoded_frames)  # [1, 1, T]

# salva come numpy 1D
clean_np    = wav.squeeze().cpu().numpy()
degraded_np = decoded.squeeze().cpu().numpy()

# allinea lunghezze
min_len = min(len(clean_np), len(degraded_np))
clean_np    = clean_np[:min_len]
degraded_np = degraded_np[:min_len]

print(f"Clean shape: {clean_np.shape}")
print(f"Degraded shape: {degraded_np.shape}")
print(f"Bandwidth impostato: 6.0 kbps (4 codebook RVQ)")

## Cella 4 — Visualizzazione: mel spectrogram, differenza e profilo energetico

Quattro grafici in una figura 2×2:
- **Mel spectrogram clean e degraded** — confronto visivo della perdita di dettaglio
- **Differenza in dB** (colormap `RdBu_r`) — evidenzia dove EnCodec aggiunge/rimuove energia
- **Profilo energetico medio per bin mel** — due curve sovrapposte mostrano il calo nelle alte frequenze

La figura viene salvata come `proof_of_concept.png` per documentazione.

In [ ]:
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt

N_MELS  = 128
N_FFT   = 1024
HOP     = 256
SR      = TARGET_SR

def compute_mel_db(audio, sr=SR, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP):
    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_mels=n_mels, n_fft=n_fft, hop_length=hop_length
    )
    return librosa.power_to_db(mel, ref=np.max)

mel_clean    = compute_mel_db(clean_np)
mel_degraded = compute_mel_db(degraded_np)
mel_diff     = mel_degraded - mel_clean   # differenza in dB

energy_clean    = mel_clean.mean(axis=1)     # [128]
energy_degraded = mel_degraded.mean(axis=1)  # [128]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("Proof of Concept — EnCodec 24kHz @ 6kbps", fontsize=14, fontweight="bold")

# 1. Mel clean
img1 = librosa.display.specshow(
    mel_clean, sr=SR, hop_length=HOP, x_axis="time", y_axis="mel", ax=axes[0, 0]
)
axes[0, 0].set_title("Mel Spectrogram — Clean")
fig.colorbar(img1, ax=axes[0, 0], format="%+2.0f dB")

# 2. Mel degraded
img2 = librosa.display.specshow(
    mel_degraded, sr=SR, hop_length=HOP, x_axis="time", y_axis="mel", ax=axes[0, 1]
)
axes[0, 1].set_title("Mel Spectrogram — Degraded (EnCodec)")
fig.colorbar(img2, ax=axes[0, 1], format="%+2.0f dB")

# 3. Differenza in dB
img3 = librosa.display.specshow(
    mel_diff, sr=SR, hop_length=HOP, x_axis="time", y_axis="mel", ax=axes[1, 0],
    cmap="RdBu_r", vmin=-20, vmax=20
)
axes[1, 0].set_title("Differenza in dB (Degraded − Clean)")
fig.colorbar(img3, ax=axes[1, 0], format="%+2.0f dB")

# 4. Profilo energetico medio
mel_freqs = librosa.mel_frequencies(n_mels=N_MELS, fmin=0, fmax=SR / 2)
axes[1, 1].plot(mel_freqs, energy_clean,    label="Clean",    linewidth=1.8)
axes[1, 1].plot(mel_freqs, energy_degraded, label="Degraded", linewidth=1.8, linestyle="--")
axes[1, 1].set_xscale("log")
axes[1, 1].set_xlabel("Frequenza (Hz)")
axes[1, 1].set_ylabel("Energia media (dB)")
axes[1, 1].set_title("Profilo energetico medio per frequenza")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("proof_of_concept.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura salvata come proof_of_concept.png")

## Cella 5 — Ascolto e metriche di qualità

Player IPython per confronto diretto clean vs degraded.
Calcoliamo due metriche:
- **LSD** (Log-Spectral Distance) — distanza media in dB tra i mel spectrogram, misura globale della distorsione
- **Perdita energetica alte frequenze** (bin mel ≥ 95) — quantifica quanto EnCodec attenua le componenti ad alta frequenza, zona più critica per la nostra UNet

In [ ]:
import numpy as np
from IPython.display import display, Audio

print("=== Audio Clean ===")
display(Audio(clean_np, rate=TARGET_SR))

print("=== Audio Degraded (EnCodec 6kbps) ===")
display(Audio(degraded_np, rate=TARGET_SR))

# --- LSD (Log-Spectral Distance) ---
lsd = np.sqrt(np.mean((mel_clean - mel_degraded) ** 2))

# --- Perdita energetica alte frequenze (bin mel >= 95) ---
HF_BIN = 95
energy_hf_clean    = mel_clean[HF_BIN:, :].mean()
energy_hf_degraded = mel_degraded[HF_BIN:, :].mean()
hf_loss_db = energy_hf_clean - energy_hf_degraded   # positivo = EnCodec attenua

print("\n=" * 40)
print(f"LSD (Log-Spectral Distance):        {lsd:.4f} dB")
print(f"Energia HF clean    (bin ≥{HF_BIN}): {energy_hf_clean:.2f} dB")
print(f"Energia HF degraded (bin ≥{HF_BIN}): {energy_hf_degraded:.2f} dB")
print(f"Perdita energetica HF:              {hf_loss_db:.2f} dB")
print("=" * 40)